# 🦀 Day 5: Testing — Unit + Integration Tests
---

Today we build a comprehensive test suite for RustKV (>80% coverage target).

- **Testing strategy**: Unit tests for storage, protocol; integration tests for server+client
- **Test helpers and fixtures**: Shared setup code
- **Property-based testing**: Concepts with proptest
- **Mocking**: Mock storage for testing server logic
- **Coverage**: `cargo tarpaulin` or `cargo llvm-cov`
- **Exercise**: Add tests for edge cases and error paths

## Testing strategy

| Layer | What to test | How |
|-------|--------------|-----|
| Storage | get/set/delete/keys | Unit tests, in-memory impl |
| Protocol | parse command, encode response | Unit tests with sample bytes |
| Server | full command flow | Integration: start server, connect client, run commands |
| Error paths | invalid input, wrong type | Unit tests asserting Err |

In [ ]:
// Unit tests for storage operations

use std::collections::HashMap;
use std::sync::RwLock;

struct MemoryStorage {
    data: RwLock<HashMap<String, String>>,
}

impl MemoryStorage {
    fn new() -> Self { Self { data: RwLock::new(HashMap::new()) } }
    fn get(&self, key: &str) -> Option<String> { self.data.read().unwrap().get(key).cloned() }
    fn set(&self, key: &str, value: &str) { self.data.write().unwrap().insert(key.to_string(), value.to_string()); }
    fn delete(&self, key: &str) -> bool { self.data.write().unwrap().remove(key).is_some() }
}

#[cfg(test)]
mod storage_tests {
    use super::*;

    #[test]
    fn test_set_get() {
        let s = MemoryStorage::new();
        s.set("foo", "bar");
        assert_eq!(s.get("foo"), Some("bar".to_string()));
    }

    #[test]
    fn test_get_missing() {
        let s = MemoryStorage::new();
        assert_eq!(s.get("nonexistent"), None);
    }

    #[test]
    fn test_delete() {
        let s = MemoryStorage::new();
        s.set("x", "y");
        assert!(s.delete("x"));
        assert_eq!(s.get("x"), None);
    }
}

println!("Storage unit tests defined.");

In [ ]:
// Unit tests for protocol parsing

fn parse_simple_command(input: &str) -> Result<Vec<&str>, String> {
    let parts: Vec<&str> = input.trim().split_whitespace().collect();
    if parts.is_empty() { return Err("empty command".to_string()); }
    Ok(parts)
}

#[cfg(test)]
mod protocol_tests {
    use super::*;

    #[test]
    fn test_parse_get() {
        let p = parse_simple_command("GET foo").unwrap();
        assert_eq!(p, ["GET", "foo"]);
    }

    #[test]
    fn test_parse_set() {
        let p = parse_simple_command("SET key value").unwrap();
        assert_eq!(p, ["SET", "key", "value"]);
    }

    #[test]
    fn test_parse_empty_err() {
        assert!(parse_simple_command("  ").is_err());
    }
}

println!("Protocol unit tests defined.");

In [ ]:
// Integration test structure (run in tests/ directory)

let integration_test = r#"
// tests/integration_test.rs
use std::io::Write;
use std::net::TcpStream;

#[test]
fn test_client_server_flow() {
    // Start server in background thread
    let handle = std::thread::spawn(|| {
        // run_server(6379)
    });
    std::thread::sleep(std::time::Duration::from_millis(100));

    let mut stream = TcpStream::connect("127.0.0.1:6379").unwrap();
    stream.write_all(b"SET foo bar\
").unwrap();
    stream.flush().unwrap();
    let mut buf = [0u8; 128];
    let n = stream.read(&mut buf).unwrap();
    assert!(String::from_utf8_lossy(&buf[..n]).contains("OK"));

    let _ = handle.join();
}
"#;

println!("Integration test structure:");
println!("{}", integration_test);

In [ ]:
// Test organization: #[cfg(test)] vs tests/ directory

let org = r#"
// In src/lib.rs or src/storage/mod.rs:
#[cfg(test)]
mod tests {
    use super::*;
    #[test] fn test_foo() { ... }
}

// In project root, create tests/:
// tests/integration_test.rs — tests that use the crate as a dependency
// tests/helpers/mod.rs — shared fixtures, spawn_server helper

// Coverage: cargo install cargo-tarpaulin
// cargo tarpaulin --out Html
"#;

println!("{}", org);

## 📝 Exercise: Add tests for edge cases and error paths

1. Test empty key, key too long, value too large
2. Test WRONGTYPE on List/Hash commands
3. Test LRANGE with negative indices, empty list
4. Test concurrent get/set from multiple threads

In [ ]:
// YOUR CODE HERE
// Add edge case and error path tests

// #[test] fn test_empty_key_rejected() { ... }
// #[test] fn test_lrange_negative_indices() { ... }
// #[test] fn test_lpush_on_string_wrongtype() { ... }

println!("Add edge case and error path tests.");

## 🎯 Key Takeaways

1. Unit tests: storage, protocol — fast, isolated, in `#[cfg(test)]` modules
2. Integration tests: start server, connect client, run commands — in `tests/` directory
3. Test helpers: shared fixtures, `spawn_server()` for integration tests
4. Mock storage: implement `Storage` trait with in-memory stub for testing server logic
5. Coverage: `cargo tarpaulin` or `cargo llvm-cov` — aim for >80%

---
**Tomorrow:** CLI interface